# UC16 — Mejora de Experiencia de Cliente (CX) en Tiempo Real

Detección de frustración, urgencia y riesgo de abandono en llamadas con alertas automáticas.

**Valor de negocio**: Menos churn + mejor NPS mediante intervención proactiva.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS CX_TIEMPO_REAL_DB;
USE SCHEMA CX_TIEMPO_REAL_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Llamadas con Diferentes Niveles de Emoción

In [ ]:
CREATE OR REPLACE TABLE llamadas_cx AS
SELECT 'CALL' || LPAD(SEQ4(),5,'0') AS llamada_id,
    'AGT' || LPAD(MOD(SEQ4(),15)+1,3,'0') AS agente_id,
    'CLI' || LPAD(UNIFORM(1,800,RANDOM()),5,'0') AS cliente_id,
    DATEADD(minute,-UNIFORM(1,4320,RANDOM()),CURRENT_TIMESTAMP()) AS timestamp_llamada,
    UNIFORM(60,600,RANDOM()) AS duracion_seg,
    CASE MOD(SEQ4(),8)
        WHEN 0 THEN 'Esto es vergonzoso. Llevo 3 meses esperando que me paguen el siniestro. He llamado 8 veces y siempre me dicen lo mismo. Voy a poner una reclamación en la DGSFP y una reseña negativa en todas las redes.'
        WHEN 1 THEN 'Estoy desesperado. El agua sigue entrando en mi casa y nadie viene a arreglarlo. Mis hijos no pueden dormir en su habitación. Necesito una solución HOY.'
        WHEN 2 THEN 'Mire, ya estoy mirando precios en otras compañías. Si no me solucionan esto en esta llamada, me cambio de seguro esta misma tarde.'
        WHEN 3 THEN 'No estoy contento con cómo se ha gestionado mi caso, pero entiendo que hay procesos. Solo pido que me mantengan informado.'
        WHEN 4 THEN 'Necesito información sobre mi póliza. ¿Me pueden decir qué coberturas tengo exactamente?'
        WHEN 5 THEN 'Quería felicitarles. El perito vino al día siguiente y en una semana ya tenía todo resuelto. Excelente servicio.'
        WHEN 6 THEN 'Es la cuarta vez que llamo por lo mismo. Cada vez me atiende alguien diferente y tengo que explicar todo desde cero. Esto es una pérdida de tiempo total.'
        ELSE 'Hola, solo quería confirmar que he recibido el pago del siniestro. Todo correcto, gracias.'
    END AS transcripcion,
    UNIFORM(1, 10, RANDOM()) AS nps_historico
FROM TABLE(GENERATOR(ROWCOUNT=>800));

SELECT * FROM llamadas_cx LIMIT 10;

---

## Paso 3: Análisis de Sentimiento y Emociones

In [ ]:
CREATE OR REPLACE TABLE analisis_cx AS
SELECT l.*,
    SNOWFLAKE.CORTEX.SENTIMENT(transcripcion) AS sentimiento_score,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Analiza la emoción y riesgo de este cliente de seguros. Responde SOLO con JSON: {"emocion_primaria": "ira/frustración/ansiedad/neutral/satisfacción", "nivel_urgencia": "critica/alta/media/baja", "riesgo_abandono": "alto/medio/bajo", "puntos_friccion": ["punto1"], "requiere_escalado": true/false}.\n\nTexto: ' || transcripcion
    ) AS analisis_raw
FROM llamadas_cx l;

SELECT llamada_id, sentimiento_score, LEFT(analisis_raw,200) AS analisis FROM analisis_cx LIMIT 10;

---

## Paso 4: Generar Alertas y Clasificar Prioridad

In [ ]:
CREATE OR REPLACE TABLE alertas_cx AS
SELECT llamada_id, agente_id, cliente_id, timestamp_llamada, sentimiento_score, nps_historico,
    TRY_PARSE_JSON(analisis_raw) AS a,
    COALESCE(a['emocion_primaria']::VARCHAR, 'neutral') AS emocion,
    COALESCE(a['nivel_urgencia']::VARCHAR, 'media') AS urgencia,
    COALESCE(a['riesgo_abandono']::VARCHAR, 'bajo') AS riesgo_abandono,
    COALESCE(a['requiere_escalado']::BOOLEAN, FALSE) AS requiere_escalado,
    CASE WHEN sentimiento_score < -0.5 AND nps_historico <= 4 THEN 'CRÍTICA'
         WHEN sentimiento_score < -0.3 OR COALESCE(a['riesgo_abandono']::VARCHAR,'bajo') = 'alto' THEN 'ALTA'
         WHEN sentimiento_score < 0 THEN 'MEDIA'
         ELSE 'BAJA' END AS prioridad_alerta
FROM analisis_cx;

SELECT prioridad_alerta, COUNT(*) AS alertas, ROUND(AVG(sentimiento_score),3) AS sentimiento_medio
FROM alertas_cx GROUP BY prioridad_alerta ORDER BY CASE prioridad_alerta WHEN 'CRÍTICA' THEN 1 WHEN 'ALTA' THEN 2 WHEN 'MEDIA' THEN 3 ELSE 4 END;

---

## Paso 5: Dashboard de CX en Tiempo Real

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('CX en Tiempo Real')
st.markdown('### Detección de Frustración y Riesgo — Snowflake Cortex')

df = session.table('alertas_cx').to_pandas()

c1,c2,c3,c4 = st.columns(4)
with c1: st.metric('Llamadas Analizadas', len(df))
with c2: st.metric('Alertas Críticas', len(df[df['PRIORIDAD_ALERTA']=='CRÍTICA']), delta=None)
with c3: st.metric('Riesgo Abandono Alto', len(df[df['RIESGO_ABANDONO']=='alto']))
with c4: st.metric('Sentimiento Medio', f"{df['SENTIMIENTO_SCORE'].mean():.2f}")

st.markdown('---')
st.subheader('Distribución de Alertas')
rc = df['PRIORIDAD_ALERTA'].value_counts()
st.bar_chart(pd.DataFrame({'Llamadas': rc.values}, index=rc.index))

st.markdown('---')
st.subheader('Emociones Detectadas')
em = df['EMOCION'].value_counts()
st.bar_chart(pd.DataFrame({'Llamadas': em.values}, index=em.index))

st.markdown('---')
st.subheader('Alertas Críticas y Altas')
criticas = df[df['PRIORIDAD_ALERTA'].isin(['CRÍTICA','ALTA'])].sort_values('SENTIMIENTO_SCORE')
st.dataframe(criticas[['LLAMADA_ID','CLIENTE_ID','AGENTE_ID','EMOCION','URGENCIA','RIESGO_ABANDONO','SENTIMIENTO_SCORE','NPS_HISTORICO','PRIORIDAD_ALERTA']].head(50), use_container_width=True, height=400)
st.caption('Desarrollado con Snowflake Cortex AI + Streamlit')

---

## Paso 6: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS CX_TIEMPO_REAL_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;